<a href="https://colab.research.google.com/github/amah67/flyrankmlintern/blob/main/notebooks/02_your_first_readable_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 2 — The model is just a rule you can read

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amah67/mlintern/blob/main/notebooks/02_your_first_readable_model.ipynb?flush_cache=true)

You'll:
1. Write a **1-line hand rule** and rank pages with it.
2. Fit a **depth-2 decision tree** and `print` it — the model "learned" a readable if/else. Then compare — where does it beat your rule, and where doesn’t it?
3. See **why you never feed the outcome back in** — that's leakage.

The payoff isn't a high score. It's: *my intuition was rough, the model found the real signal, and I can read exactly what it found.*

## 0. Setup (Colab or local)

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# The label: a page is 'declining' when its recent trend is down. Simple, honest starter label.
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(df.shape[0], "pages |  declining rate:", round(df["is_declining_label"].mean(), 3))

30000 pages |  declining rate: 0.542


## 1. A rule you write by hand: `stale x visible`
Intuition: a page worth reviewing is one that is **stale** (not updated in a while) **and** still **visible** (getting impressions). Rank those by how much exposure they have.

In [2]:
stale   = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["hand_rule_score"] = stale * visible * df["impressions_90d"]

top10 = df.sort_values("hand_rule_score", ascending=False).head(10)
top10[["impressions_90d", "days_since_last_update", "avg_position", "ctr", "trend_direction"]]

,impressions_90d,days_since_last_update,avg_position,ctr,trend_direction
16751,61678,194,19.7,0.15,down
16514,59472,194,24.8,0.13,down
7021,25715,194,22.2,0.23,down
21268,13299,193,10.5,0.49,down
11489,7812,194,39.0,0.01,down
12045,7558,193,17.9,0.20,down
698,4590,194,31.0,0.00,down
5327,4556,194,16.4,0.33,down
26810,4429,194,25.3,0.38,down
20837,1697,193,15.8,0.12,down


We need a way to score any ranking. **Precision@K** = of the top K pages a ranking flags, what fraction are actually declining.

In [3]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

y = df["is_declining_label"].values
for k in (20, 50):
    print(f"Hand rule  Precision@{k}: {precision_at_k(df['hand_rule_score'], y, k):.3f}")

Hand rule  Precision@20: 0.900
Hand rule  Precision@50: 0.680


## 2. Let a model learn the rule — then read it
A **depth-2 decision tree** can only ask 3 yes/no questions. That constraint is the point: whatever it learns, you can read.

We give it a few **pre-decision** signals — never product flags.

In [4]:
from sklearn.tree import DecisionTreeClassifier, export_text

features = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)

tree = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
tree.fit(X, y)

print(export_text(tree, feature_names=features))

|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   |   |--- class: 0
|   |--- avg_position >  0.75
|   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 312.50
|   |   |--- class: 1
|   |--- content_age_days >  312.50
|   |   |--- class: 0



That printout **is** the model — a human-readable if/else. Now rank pages by the tree's probability and score it the same way.

In [5]:
tree_score = tree.predict_proba(X)[:, 1]
for k in (20, 50):
    hr = precision_at_k(df["hand_rule_score"], y, k)
    tr = precision_at_k(tree_score, y, k)
    print(f"Precision@{k}:  hand rule {hr:.3f}   vs   tree {tr:.3f}")

Precision@20:  hand rule 0.900   vs   tree 0.550
Precision@50:  hand rule 0.680   vs   tree 0.600


Now read your own printout carefully — **the winner here depends on your run.** A depth-2 tree can only give four different scores (one per leaf), so the "top 50" is mostly one big block of tied pages, and different library versions break those ties differently. On some stacks the tree wins at Precision@50; on others the hand rule holds both. **Both results are real.** The stable lesson: a sharp human rule can be excellent at the very top of the list; a model's advantage — when it shows up — appears deeper, where simple rules run out of signal; and any comparison built on heavily tied scores is fragile. Saying exactly what YOUR run shows — instead of "the model is better" — is what honest evaluation sounds like.

## 3. Why you can't feed the outcome back in
Your label is `trend_direction == "down"`, and `trend_pct` is the exact percentage change that bucket is computed from — so it **is** the answer in disguise. Watch what happens if you feed it in as a feature:

In [6]:
X_leaky = df[features + ["trend_pct"]].replace([np.inf, -np.inf], np.nan).fillna(0)
leaky = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42).fit(X_leaky, y)
print(f"'Leaky' tree Precision@50: {precision_at_k(leaky.predict_proba(X_leaky)[:,1], y, 50):.3f}  <- looks amazing")
print(export_text(leaky, feature_names=features + ["trend_pct"]))

'Leaky' tree Precision@50: 1.000  <- looks amazing
|--- trend_pct <= -20.05
|   |--- word_count <= 212.00
|   |   |--- class: 1
|   |--- word_count >  212.00
|   |   |--- class: 1
|--- trend_pct >  -20.05
|   |--- trend_pct <= -19.95
|   |   |--- class: 0
|   |--- trend_pct >  -19.95
|   |   |--- class: 0



The tree just split on `trend_pct` and nailed the label — because the label is **derived from** `trend_pct`. That's **leakage**: the feature is the answer in disguise, and it teaches you nothing.

That's also why the starter data ships **only observable signals** — the product's own decision flags (health scores, "needs CTR fix", and so on) aren't included, so you can't accidentally train on them. You build from what was knowable *before* the outcome.

> Rule of thumb: if a feature would only be known *because someone already made the decision you're predicting*, it leaks. Leave it out.

## 4. 🔧 Your turn
- Change `max_depth` to 3 or 4 — does Precision@50 improve? Can you still read the tree?
- Swap in different features (drop `impressions_90d`, add `engagement_rate`). What does the tree choose to split on first?
- **Important caveat:** we scored *in-sample* here for teaching. The real pipeline uses **client-holdout** validation (`scripts/03_train_model.py`) so a client's pages never appear in both train and test. Re-run your comparison with a train/test split and see if the gap holds.

Write your experiment below.

In [8]:
# Your experiment here

from sklearn.tree import DecisionTreeClassifier, export_text

features = ["content_age_days", "days_since_last_update", "engagement_rate",
            "avg_position", "ctr", "word_count"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)

tree = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42)
tree.fit(X, y)

print(export_text(tree, feature_names=features))

tree_score = tree.predict_proba(X)[:, 1]
for k in (20, 50):
    hr = precision_at_k(df["hand_rule_score"], y, k)
    tr = precision_at_k(tree_score, y, k)
    print(f"Precision@{k}:  hand rule {hr:.3f}   vs   tree {tr:.3f}")

|--- avg_position <= 0.55
|   |--- avg_position <= 0.15
|   |   |--- word_count <= 669.50
|   |   |   |--- class: 0
|   |   |--- word_count >  669.50
|   |   |   |--- class: 0
|   |--- avg_position >  0.15
|   |   |--- days_since_last_update <= 62.00
|   |   |   |--- class: 0
|   |   |--- days_since_last_update >  62.00
|   |   |   |--- class: 1
|--- avg_position >  0.55
|   |--- content_age_days <= 287.50
|   |   |--- ctr <= 0.33
|   |   |   |--- class: 1
|   |   |--- ctr >  0.33
|   |   |   |--- class: 1
|   |--- content_age_days >  287.50
|   |   |--- avg_position <= 38.45
|   |   |   |--- class: 0
|   |   |--- avg_position >  38.45
|   |   |   |--- class: 0

Precision@20:  hand rule 0.900   vs   tree 0.850
Precision@50:  hand rule 0.680   vs   tree 0.740


The hand rule does a slightly better job than the decision tree when comparing it against a smaller amount of items. However when going against a broader amount of pages the decision tree outperforms the hand rule.

### Re-running comparison with Train/Test Split

To properly evaluate the model's generalization capabilities, we split the data into training and testing sets. The model will be trained only on the training data and evaluated on the unseen test data. We'll use `stratify=y` to ensure that the proportion of declining pages is maintained in both the training and test sets.

In [9]:
from sklearn.model_selection import train_test_split

# Split data into training and testing sets
# We use stratify=y to ensure the proportion of 'declining' labels is similar in train and test sets.
X_train, X_test, y_train, y_test, hand_rule_score_train, hand_rule_score_test = train_test_split(
    X, y, df['hand_rule_score'], test_size=0.3, random_state=42, stratify=y
)

print(f"Train set size: {len(X_train)} samples")
print(f"Test set size: {len(X_test)} samples")
print(f"Train declining rate: {y_train.mean():.3f}")
print(f"Test declining rate: {y_test.mean():.3f}")


Train set size: 21000 samples
Test set size: 9000 samples
Train declining rate: 0.542
Test declining rate: 0.542


Now, we will train the Decision Tree only on the training data and then evaluate its performance on the unseen test data. This is a more robust way to assess how well the model would perform in a real-world scenario on new pages.

In [10]:
# Re-train the decision tree on the training data
tree_split = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42)
tree_split.fit(X_train, y_train)

# Get predictions (probabilities) on the TEST data
tree_score_test = tree_split.predict_proba(X_test)[:, 1]

print("\n--- Decision Tree (trained on train, evaluated on test) ---")
print(export_text(tree_split, feature_names=features))

# Compare Precision@K for hand rule vs. tree on the TEST data
for k in (20, 50):
    hr_test = precision_at_k(hand_rule_score_test, y_test, k)
    tr_test = precision_at_k(tree_score_test, y_test, k)
    print(f"Precision@{k} (on TEST set):  hand rule {hr_test:.3f}   vs   tree {tr_test:.3f}")



--- Decision Tree (trained on train, evaluated on test) ---
|--- avg_position <= 0.55
|   |--- avg_position <= 0.15
|   |   |--- engagement_rate <= 75.00
|   |   |   |--- class: 0
|   |   |--- engagement_rate >  75.00
|   |   |   |--- class: 0
|   |--- avg_position >  0.15
|   |   |--- word_count <= 687.00
|   |   |   |--- class: 1
|   |   |--- word_count >  687.00
|   |   |   |--- class: 0
|--- avg_position >  0.55
|   |--- content_age_days <= 287.50
|   |   |--- ctr <= 0.30
|   |   |   |--- class: 1
|   |   |--- ctr >  0.30
|   |   |   |--- class: 1
|   |--- content_age_days >  287.50
|   |   |--- avg_position <= 39.85
|   |   |   |--- class: 0
|   |   |--- avg_position >  39.85
|   |   |   |--- class: 0

Precision@20 (on TEST set):  hand rule 0.700   vs   tree 0.450
Precision@50 (on TEST set):  hand rule 0.580   vs   tree 0.680


Both results are lower then evaluated with a test set and this is good news due to the fact that it's working on unseen data.

For Precision@20 the hand rule is significantly better than using the decision tree. This shows that the hand rule is more effective when identifying the top declining pages, even on unseen data.

For Percision@50 the tree outperforms the hand rule. This shows that when looking and more broad ranking the decision tree does a better job in finding pages that are truly declining.

### Save your work
**Colab:** *File → Save a copy in GitHub* (your submission) and *File → Save a copy in Drive*.

You now have the two core reflexes of applied ML: **discover before you model**, and **prefer a model you can read and can't fool**. That's the whole foundation the capstone builds on.